# Session 7 — Frontend: HTML, CSS & JavaScript
## Empire & Ink: AI Mughal Art Studio

**What you'll build:** The complete browser interface — register, login, generate art with style cards, and a gallery with lightbox, download, and delete.

---

## Setup

In [ ]:
!pip install jinja2

## Theory 1 — HTML Skeleton & Tailwind CSS

**HTML** defines structure. **Tailwind** is utility-first CSS — you style elements with short class names:

```html
<!-- Traditional CSS approach -->
<button class="my-btn">Click</button>
<style>.my-btn { background: gold; padding: 8px 16px; border-radius: 4px; }</style>

<!-- Tailwind approach -->
<button class="bg-yellow-400 px-4 py-2 rounded">Click</button>
```

Key patterns: `text-xl` = large text, `mt-4` = margin-top 1rem, `flex` = flexbox, `hidden` = display none, `group-hover:opacity-100` = show on parent hover.

In [ ]:
from jinja2 import Environment, BaseLoader

def render(template_str, **context):
    env = Environment(loader=BaseLoader())
    return env.from_string(template_str).render(**context)

html = render(
    "<h1 class='text-3xl font-bold text-yellow-400'>{{ title }}</h1><p>{{ subtitle }}</p>",
    title="Empire & Ink",
    subtitle="AI Mughal Art Studio"
)
print(html)

## Theory 2 — JavaScript & the Fetch API

**JavaScript** makes web pages interactive. The **Fetch API** sends HTTP requests without page refresh:

```javascript
const response = await fetch('/api/gallery', {
    headers: { 'Authorization': 'Bearer ' + token }
});
const data = await response.json();
console.log(data.items);
```

**localStorage** persists data across page loads:
```javascript
localStorage.setItem('access_token', token);      // save
const token = localStorage.getItem('access_token'); // load
localStorage.removeItem('access_token');            // delete
```

In [ ]:
# Python equivalent of localStorage + Auth pattern
class LocalStorage:
    def __init__(self): self._store = {}
    def setItem(self, key, val): self._store[key] = val
    def getItem(self, key): return self._store.get(key)
    def removeItem(self, key): self._store.pop(key, None)

class Auth:
    def __init__(self, storage): self.storage = storage
    def getToken(self):  return self.storage.getItem("access_token")
    def setSession(self, token, user_id):
        self.storage.setItem("access_token", token)
        self.storage.setItem("user_id", user_id)
    def clearSession(self):
        self.storage.removeItem("access_token")
        self.storage.removeItem("user_id")
    def requireAuth(self):
        if not self.getToken(): raise Exception("Not authenticated")

storage = LocalStorage()
auth = Auth(storage)
auth.setSession("tok-abc123", "user-001")
print("Token:", auth.getToken())
auth.clearSession()
print("After clear:", auth.getToken())

## Theory 3 — Jinja2 Template Inheritance

Templates can **extend** a base layout:

```html
<!-- base.html -->
<!DOCTYPE html><html><body>
  <nav><!-- navbar --></nav>
  {% block content %}{% endblock %}
</body></html>

<!-- gallery.html -->
{% extends 'base.html' %}
{% block content %}
  {% for item in items %}<div>{{ item.prompt }}</div>{% endfor %}
{% endblock %}
```

In [ ]:
from jinja2 import Environment, DictLoader

templates = {
    "base.html": (
        "<!DOCTYPE html><html><body class='bg-gray-900 text-white'>"
        "<nav class='bg-yellow-800 p-4'>"
        "  <span class='text-xl font-bold text-yellow-200'>Empire & Ink</span>"
        "</nav><main class='p-8'>"
        "{% block content %}{% endblock %}"
        "</main></body></html>"
    ),
    "gallery.html": (
        "{% extends 'base.html' %}"
        "{% block content %}"
        "<h1 class='text-3xl text-yellow-400 mb-4'>Your Gallery ({{ items|length }} items)</h1>"
        "{% for item in items %}"
        "<div class='border border-yellow-700 p-3 mb-2 rounded'>{{ item.prompt }}</div>"
        "{% endfor %}"
        "{% endblock %}"
    ),
}
env  = Environment(loader=DictLoader(templates))
html = env.get_template("gallery.html").render(items=[{"prompt": "A peacock"}, {"prompt": "A tiger"}])
print(html[:400])

---
## In-Class Exercises

### Exercise 1 — Jinja2 template rendering a paintings list

In [ ]:
from jinja2 import Environment, DictLoader

paintings = [
    {"title": "Baburnama Folio", "style": "Akbari",    "year": 1598},
    {"title": "Padshahnama",     "style": "Shahjahani", "year": 1656},
    {"title": "Imperial Hunt",   "style": "Jahangiri",  "year": 1615},
]

# YOUR CODE HERE — write a template that renders each painting as an <li> element
# showing the title, style, and year with Tailwind CSS classes
template_str = (
    "<ul class='space-y-2'>"
    "{% for p in paintings %}"
    "<!-- YOUR TEMPLATE HERE -->"
    "{% endfor %}"
    "</ul>"
)

env  = Environment(loader=DictLoader({"t.html": template_str}))
html = env.get_template("t.html").render(paintings=paintings)
print(html)

### Exercise 2 — HTML form + fetch JS

In [ ]:
# Write the HTML for the generation form and the JS submit handler.
# The form: text input for prompt, select for style (akbari/jahangiri/shahjahani), submit button.
# The JS: prevent default, read values, call fetch POST /api/generate.
# YOUR CODE HERE
form_html = (
    "<form id='generate-form'>\n"
    "  <input type='text' id='prompt-input' placeholder='Describe your painting...' "
    "class='w-full p-3 bg-gray-800 border border-yellow-700 rounded text-white'>\n"
    "  <select id='style-select' class='w-full mt-2 p-3 bg-gray-800 border border-yellow-700 rounded text-white'>\n"
    "    <option value='akbari'>Akbari</option>\n"
    "    <option value='jahangiri'>Jahangiri</option>\n"
    "    <option value='shahjahani'>Shahjahani</option>\n"
    "  </select>\n"
    "  <button type='submit' class='mt-4 w-full bg-yellow-600 text-white py-3 rounded font-bold'>\n"
    "    Generate Art\n"
    "  </button>\n"
    "</form>\n"
    "<script>\n"
    "// YOUR JS HERE: addEventListener submit -> preventDefault -> fetch POST /api/generate\n"
    "</script>"
)
print(form_html)

### Exercise 3 — Auth JS object (getToken / setSession / clearSession / requireAuth)

In [ ]:
# Write the JS Auth object as a Python string — you'll paste it into static/app.js
auth_js = (
    "const Auth = {\n"
    "  getToken() { return localStorage.getItem('access_token'); },\n"
    "  setSession(accessToken, userId, email) {\n"
    "    localStorage.setItem('access_token', accessToken);\n"
    "    localStorage.setItem('user_id', userId);\n"
    "    localStorage.setItem('user_email', email);\n"
    "  },\n"
    "  clearSession() {\n"
    "    // YOUR CODE HERE — remove access_token, user_id, user_email\n"
    "  },\n"
    "  requireAuth() {\n"
    "    if (!this.getToken()) { window.location.href = '/login'; return false; }\n"
    "    return true;\n"
    "  },\n"
    "  isAuthenticated() {\n"
    "    // YOUR CODE HERE — return true if token exists\n"
    "  }\n"
    "};\n"
)
print(auth_js)

### Exercise 4 — Gallery card HTML with hover overlay

In [ ]:
# Build the HTML for a gallery card with a hover overlay showing prompt + download button.
card_html = (
    "<div class='relative group overflow-hidden rounded-lg border border-yellow-800/50 cursor-pointer'\n"
    "     onclick='openLightbox(item.image_url, item.prompt)'>\n"
    "  <img src='{{ item.image_url }}' alt='{{ item.prompt }}'\n"
    "       class='w-full aspect-square object-cover transition-transform duration-300 group-hover:scale-105'>\n"
    "  <!-- Hover overlay — YOUR CODE HERE -->\n"
    "  <!-- div: opacity-0 by default, group-hover:opacity-100, flex col justify-end p-4 -->\n"
    "  <div class='absolute inset-0 bg-black/70 opacity-0 group-hover:opacity-100\n"
    "              transition-opacity duration-300 flex flex-col justify-end p-4'>\n"
    "    <p class='text-white text-sm line-clamp-2'>{{ item.prompt }}</p>\n"
    "    <button onclick='downloadImage(event, item.image_url)'\n"
    "            class='mt-2 bg-yellow-600 text-white text-xs py-1 px-3 rounded hover:bg-yellow-500'>\n"
    "      Download\n"
    "    </button>\n"
    "  </div>\n"
    "</div>"
)
print(card_html[:500])

### Exercise 5 — apiFetch() wrapper with Bearer token

In [ ]:
api_fetch_js = (
    "async function apiFetch(path, options = {}) {\n"
    "  const token = Auth.getToken();\n"
    "  const headers = { 'Content-Type': 'application/json', ...options.headers };\n"
    "  // YOUR CODE HERE — if token exists, add: headers['Authorization'] = 'Bearer ' + token\n"
    "  const response = await fetch(path, { ...options, headers });\n"
    "  if (response.status === 401) {\n"
    "    Auth.clearSession();\n"
    "    window.location.href = '/login';\n"
    "    return;\n"
    "  }\n"
    "  if (!response.ok) {\n"
    "    const err = await response.json().catch(() => ({ detail: 'Unknown error' }));\n"
    "    throw new Error(err.detail || 'Request failed');\n"
    "  }\n"
    "  return response.json();\n"
    "}\n"
)
print(api_fetch_js)

---
## Post-Class Exercises

### Challenge 1 — Complete login form with client-side validation

In [ ]:
# Write HTML + JS for a login form with validation:
# - Email must contain "@"
# - Password must be >= 8 characters
# - Show errors in a #login-error div
# - On success: call apiFetch POST /api/auth/login, then Auth.setSession(), redirect to /
# YOUR CODE HERE
login_form_sketch = (
    "<form id='login-form' class='max-w-md mx-auto mt-16 space-y-4'>\n"
    "  <h1 class='text-3xl font-bold text-yellow-400 text-center'>Sign In</h1>\n"
    "  <input type='email'    id='login-email'    required placeholder='Email'>\n"
    "  <input type='password' id='login-password' required minlength='8' placeholder='Password'>\n"
    "  <div id='login-error' class='text-red-400 text-sm hidden'></div>\n"
    "  <button type='submit' class='w-full bg-yellow-600 text-white py-3 rounded'>Sign In</button>\n"
    "</form>\n"
    "<script>\n"
    "// YOUR JS: submit handler with validation -> apiFetch -> Auth.setSession() -> redirect\n"
    "</script>"
)
print(login_form_sketch)

### Challenge 2 — downloadImage() via fetch → blob

In [ ]:
# Write the JS downloadImage(event, imageUrl) function:
# 1. event.stopPropagation()
# 2. fetch(imageUrl) -> response.blob()
# 3. URL.createObjectURL(blob)
# 4. Create <a> with href + download filename, click it
# 5. URL.revokeObjectURL() to clean up
download_js = (
    "async function downloadImage(event, imageUrl) {\n"
    "  event.stopPropagation();\n"
    "  try {\n"
    "    const response = await fetch(imageUrl);\n"
    "    const blob = await response.blob();\n"
    "    const url = URL.createObjectURL(blob);\n"
    "    const a = document.createElement('a');\n"
    "    a.href = url;\n"
    "    // YOUR CODE HERE — set a.download to a filename like empire-ink-timestamp.png\n"
    "    document.body.appendChild(a);\n"
    "    a.click();\n"
    "    document.body.removeChild(a);\n"
    "    URL.revokeObjectURL(url);\n"
    "  } catch (err) { console.error('Download failed:', err); }\n"
    "}\n"
)
print(download_js)

### Challenge 3 — Lightbox HTML structure

In [ ]:
# Build the lightbox overlay HTML and openLightbox / closeLightbox JS functions.
lightbox_html = (
    "<div id='lightbox' class='fixed inset-0 z-50 hidden bg-black/90 flex items-center justify-center p-4'\n"
    "     onclick='closeLightbox()'>\n"
    "  <div class='relative max-w-3xl w-full' onclick='event.stopPropagation()'>\n"
    "    <button onclick='closeLightbox()' class='absolute -top-10 right-0 text-white text-2xl'>&times;</button>\n"
    "    <img id='lightbox-image' src='' alt='' class='w-full rounded-lg border border-yellow-800'>\n"
    "    <p id='lightbox-caption' class='mt-4 text-gray-300 text-sm text-center'></p>\n"
    "    <div class='flex justify-center mt-4'>\n"
    "      <button onclick='downloadImage(event, lightboxImage.src)'\n"
    "              class='bg-yellow-600 text-white px-6 py-2 rounded font-semibold'>Download</button>\n"
    "    </div>\n"
    "  </div>\n"
    "</div>\n"
    "<script>\n"
    "function openLightbox(imageUrl, caption) {\n"
    "  // YOUR CODE HERE — set src + caption, remove 'hidden' from #lightbox\n"
    "}\n"
    "function closeLightbox() {\n"
    "  document.getElementById('lightbox').classList.add('hidden');\n"
    "}\n"
    "</script>"
)
print(lightbox_html[:500])

### Challenge 4 — Shimmer skeleton CSS + JS

In [ ]:
# Write the .skeleton CSS class with shimmer animation
# and showSkeletons(count) / clearSkeletons() JS functions.
shimmer_css = (
    "@keyframes shimmer {\n"
    "  0%   { background-position: -200% 0; }\n"
    "  100% { background-position:  200% 0; }\n"
    "}\n"
    ".skeleton {\n"
    "  background: linear-gradient(90deg, #2d2416 25%, #3d3220 50%, #2d2416 75%);\n"
    "  background-size: 200% 100%;\n"
    "  animation: shimmer 1.5s infinite;\n"
    "  border-radius: 0.5rem;\n"
    "}\n"
)
skeleton_js = (
    "function showSkeletons(count = 6) {\n"
    "  const grid = document.getElementById('gallery-grid');\n"
    "  for (let i = 0; i < count; i++) {\n"
    "    const el = document.createElement('div');\n"
    "    el.className = 'skeleton aspect-square w-full';\n"
    "    grid.appendChild(el);\n"
    "  }\n"
    "}\n"
    "function clearSkeletons() {\n"
    "  // YOUR CODE HERE — remove all .skeleton elements from #gallery-grid\n"
    "}\n"
)
print("CSS:")
print(shimmer_css)
print("JS:")
print(skeleton_js)

---
## What you built today

- Rendered dynamic HTML from Python data using Jinja2 template inheritance
- Wrote a JavaScript Auth object managing tokens in localStorage
- Built a gallery card with hover overlay using Tailwind CSS utility classes
- Created apiFetch() to send authenticated requests with Bearer tokens

**Next session:** Session 8 — Deployment & Launch — ship Empire & Ink live to the internet!